In [2]:
# -----------------------------
# Unified M-NMF Embedding Pipeline with Seed Support, Split Support, and Average Execution Time
# -----------------------------

import os
import pandas as pd
import numpy as np
import sys
import time
from torch_geometric.datasets import Planetoid, WikiCS, Amazon

# Import modified M-NMF function that returns embeddings
sys.path.append("./src")
from main_get_emb import create_and_run_model
from param_parser import parameter_parser

# -----------------------------
# Step 0: Dataset Configuration
# -----------------------------
#DATASET_CONFIGS = [
    #{"name": "Cora", "loader": Planetoid},
    #{"name": "CiteSeer", "loader": Planetoid},
    #{"name": "PubMed", "loader": Planetoid},
    #{"name": "WikiCS", "loader": WikiCS},
    #{"name": "AmazonPhotos", "loader": Amazon},  # PyG expects "AmazonPhotos"
#]

DATASET_CONFIGS = [
    #{"name": "Cora", "loader": Planetoid, "pyg_name": "Cora"},
    #{"name": "CiteSeer", "loader": Planetoid, "pyg_name": "CiteSeer"},
    #{"name": "PubMed", "loader": Planetoid, "pyg_name": "PubMed"},
    #{"name": "WikiCS", "loader": WikiCS, "pyg_name": None},  # WikiCS doesn’t need a name
    {"name": "AmazonPhotos", "loader": Amazon, "pyg_name": "photo"},  # Use "photos"
]


SEEDS = [42, 46, 123, 2025, 999]
SPLITS = ['30_70', '70_30']

BASE_DIR = "."
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

# -----------------------------
# Step 1: Iterate over datasets
# -----------------------------
for ds in DATASET_CONFIGS:
    dataset_name = ds["name"]
    loader = ds["loader"]

    print(f"\nProcessing dataset: {dataset_name}")

    # Paths
    data_dir = os.path.join(BASE_DIR, "data", dataset_name)
    emb_base_dir = os.path.join(OUTPUT_DIR, "embeddings", dataset_name)
    cluster_dir = os.path.join(OUTPUT_DIR, "cluster_means", dataset_name)
    assign_dir = os.path.join(OUTPUT_DIR, "assignments", dataset_name)
    log_dir = os.path.join(OUTPUT_DIR, "logs", dataset_name)
    time_dir = os.path.join(OUTPUT_DIR, "execution_times")

    os.makedirs(data_dir, exist_ok=True)
    os.makedirs(cluster_dir, exist_ok=True)
    os.makedirs(assign_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(time_dir, exist_ok=True)

    # Load Dataset
    #if loader == Planetoid:
    #    dataset = loader(root=data_dir, name=dataset_name)
    #elif loader == WikiCS:
    #    dataset = loader(root=data_dir)
    #elif loader == Amazon:
    #    dataset = loader(root=data_dir, name=dataset_name)
    #else:
    #    raise NotImplementedError(f"Loader for {dataset_name} not implemented!")
    
    if loader == Planetoid or loader == Amazon:
        dataset = loader(root=data_dir, name=ds["pyg_name"])
    elif loader == WikiCS:
        dataset = loader(root=data_dir)
    else:
        raise NotImplementedError(f"Loader for {dataset_name} not implemented!")

    data = dataset[0]

    # Save edges as CSV
    if hasattr(data, 'edge_index'):
        edges = data.edge_index.t().numpy()
    else:
        adj = data.adj_t.to_dense()
        edges = np.array(np.nonzero(adj)).T

    edge_df = pd.DataFrame(edges, columns=['source', 'target'])
    edge_csv = os.path.join(data_dir, f"{dataset_name}_edges.csv")
    edge_df.to_csv(edge_csv, index=False)
    print(f"Edge list saved to {edge_csv}")

    for split in SPLITS:
        print(f"\nProcessing split: {split}")

        emb_dir = os.path.join(emb_base_dir, split)
        os.makedirs(emb_dir, exist_ok=True)

        if split == '30_70':
            total_execution_time = 0.0

            for seed in SEEDS:
                print(f"\nRunning seed: {seed}")

                sys.argv = ['']  # Clear command-line arguments

                args = parameter_parser()
                args.input = edge_csv
                args.embedding_output = os.path.join(
                    emb_dir, f"{dataset_name}_{split}_seed{seed}_mnmf.pkl"
                )
                args.cluster_mean_output = os.path.join(cluster_dir, f"{dataset_name}_seed{seed}_clusters.csv")
                args.assignment_output = os.path.join(assign_dir, f"{dataset_name}_seed{seed}_assignment.json")
                args.log_output = os.path.join(log_dir, f"{dataset_name}_seed{seed}_log.json")
                args.dump_matrices = True
                args.dimensions = 150
                np.random.seed(seed)

                start_time = time.time()
                embeddings = create_and_run_model(args)
                end_time = time.time()

                execution_duration = end_time - start_time
                total_execution_time += execution_duration

                print(f"Seed {seed} execution time: {execution_duration:.2f} seconds")

                emb_pkl_path = os.path.join(emb_dir, f"{dataset_name}_{split}_seed{seed}_mnmf.pkl")
                pd.to_pickle(embeddings, emb_pkl_path)
                print(f"Embeddings saved as .pkl at {emb_pkl_path}")

                # Copy embedding to 70_30 folder
                emb_dir_70_30 = os.path.join(emb_base_dir, '70_30')
                os.makedirs(emb_dir_70_30, exist_ok=True)
                emb_pkl_70_30_path = os.path.join(emb_dir_70_30, f"{dataset_name}_70_30_seed{seed}_mnmf.pkl")
                pd.to_pickle(embeddings, emb_pkl_70_30_path)
                print(f"Copied embedding to 70_30 split: {emb_pkl_70_30_path}")

            avg_execution_time = total_execution_time / len(SEEDS)
            print(f"\nAverage execution time for {dataset_name} (30_70): {avg_execution_time:.2f} seconds")

            # Save average execution time once
            time_df = pd.DataFrame({
                "dataset": [dataset_name],
                "split": ['30_70'],
                "average_execution_time_sec": [avg_execution_time]
            })
            time_csv_path = os.path.join(time_dir, f"{dataset_name}_avg_execution_time.csv")
            time_df.to_csv(time_csv_path, index=False)
            print(f"Average execution time saved to {time_csv_path}")

        else:
            # Skip model run for '70_30' split
            print(f"Skipping model run for split {split}, since it's a copy from 30_70.")


    print(f"{dataset_name} processing completed!\n")



Processing dataset: AmazonPhoto


Processing...
C:\Users\user\anaconda3\envs\mnmf\lib\site-packages\torch_geometric\io\npz.py:21: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at  ..\torch\csrc\utils\tensor_new.cpp:201.)
  edge_index = torch.tensor([adj.row, adj.col], dtype=torch.long)
Done!


Edge list saved to .\data\AmazonPhoto\AmazonPhoto_edges.csv

Processing split: 30_70

Running seed: 42
+---------------------+--------------------------------------------------------+
|        Input        |        .\data\AmazonPhoto\AmazonPhoto_edges.csv        |
+=====================+========================================================+
| Embedding output    | .\output\embeddings\AmazonPhoto\30_70\AmazonPhoto_30_7 |
|                     | 0_seed42_mnmf.pkl                                      |
+---------------------+--------------------------------------------------------+
| Cluster mean output | .\output\cluster_means\AmazonPhoto\AmazonPhoto_seed42_ |
|                     | clusters.csv                                           |
+---------------------+--------------------------------------------------------+
| Log output          | .\output\logs\AmazonPhoto\AmazonPhoto_seed42_log.json  |
+---------------------+--------------------------------------------------------+
| Assi

100%|██████████████████████████████████████████████████████████████████████████████| 7535/7535 [01:32<00:00, 81.54it/s]


Modularity calculation.



...

Seed 2025 execution time: 324.43 seconds
Embeddings saved as .pkl at .\output\embeddings\AmazonPhoto\30_70\AmazonPhoto_30_70_seed2025_mnmf.pkl
Copied embedding to 70_30 split: .\output\embeddings\AmazonPhoto\70_30\AmazonPhoto_70_30_seed2025_mnmf.pkl

Running seed: 999
+---------------------+--------------------------------------------------------+
|        Input        |        .\data\AmazonPhoto\AmazonPhoto_edges.csv        |
+=====================+========================================================+
| Embedding output    | .\output\embeddings\AmazonPhoto\30_70\AmazonPhoto_30_7 |
|                     | 0_seed999_mnmf.pkl                                     |
+---------------------+--------------------------------------------------------+
| Cluster mean output | .\output\cluster_means\AmazonPhoto\AmazonPhoto_seed999 |
|                     | _clusters.csv                                          |
+---------------------+--------------------------------------------------------+
| 

100%|██████████████████████████████████████████████████████████████████████████████| 7535/7535 [01:28<00:00, 85.57it/s]


Modularity calculation.



100%|████████████████████████████████████████████████████████████████████████████████| 200/200 [03:05<00:00,  1.08it/s]


Seed 999 execution time: 323.69 seconds
Embeddings saved as .pkl at .\output\embeddings\AmazonPhoto\30_70\AmazonPhoto_30_70_seed999_mnmf.pkl
Copied embedding to 70_30 split: .\output\embeddings\AmazonPhoto\70_30\AmazonPhoto_70_30_seed999_mnmf.pkl

Average execution time for AmazonPhoto (30_70): 323.31 seconds
Average execution time saved to .\output\execution_times\AmazonPhoto_avg_execution_time.csv

Processing split: 70_30
Skipping model run for split 70_30, since it's a copy from 30_70.
AmazonPhoto processing completed!

